In [ ]:
import pandas as pd
import transformers
print("ready")


In [ ]:
df_default = pd.read_csv("data/amazon_reviews_us_Digital_Video_Games_v1_00.tsv.zip", sep="\t")
df_default.shape

Pandas loaded 144724 rows and 15 columns

In [ ]:
import csv

df = pd.read_csv(
    "data/amazon_reviews_us_Digital_Video_Games_v1_00.tsv.zip",
    sep="\t",
    quoting=csv.QUOTE_NONE,
)
print("Default load:", df_default.shape)
print("Correct load:", df.shape)
print("Difference in rows:", len(df) - len(df_default))

The default load gave 144,724 rows, and the corrected load gave 145,431 rows, both with 15 columns. That's a difference of 707 reviews.

Pandas did not give any warning or error. It silently lost 707 rows because stray quote marks in some reviews made pandas think a quoted section was starting, so it merged several reviews together into one row.

This tells me that code running without errors doesn't mean the data loaded correctly. You should always check the data after loading it, for example by comparing the row count to what you expect and looking at some of the actual rows.

In [ ]:
for i, row in df.head(5).iterrows():
    print(f"--- {row['star_rating']} stars: {row['product_title']}")
    print(row["review_body"])
    print()

**What looks weird:**
- The first review is full of `<br /><br />` tags. These are HTML line breaks, not actual text the customer typed.
- There are missing spaces after periods, like "saying much.The game", probably where a line break used to be.
- Some reviews are just one word ("Awesome", "Perfect"), which gives a model very little to work with.
- Some "games" aren't games at all: an Xbox Live Gift Card and a PlayStation Plus Subscription.

**Where it came from:**
The reviews were collected from Amazon's website, which stores text as HTML. When a customer pressed Enter to start a new paragraph, the website saved it as a `<br />` tag, and that HTML got carried into the dataset instead of being converted back into normal line breaks.

In [ ]:
counts = df["star_rating"].value_counts().sort_index()
print(counts)
print()
print((counts / len(df) * 100).round(1).astype(str) + "%")

The ratings are very unbalanced. 5 stars is by far the most common (80,677 reviews, 55.5%), followed by 1 star (17.2%) and 4 stars (14.0%). 3 stars (8.0%) and 2 stars (5.3%) are rare. It's a "J-shape": people mostly review when they love or hate something.

This matters for measuring accuracy because a model could just guess "positive" for every review and still be right about 70% of the time (4 and 5 stars together). High accuracy on this data could hide that the model is bad at spotting negative or mixed reviews. To really test a model, we need enough examples of every rating, not just the common ones.

In [ ]:
df["product_title"].value_counts().head(15)

Most of the top "games" aren't games. The four most-reviewed products are a PlayStation Network Card, an Xbox Live Subscription, a PlayStation Plus Subscription, and an Xbox Live Gift Card, plus an Xbox 360 Live Points Card at #6. Together these gift cards and subscriptions have over 32,000 reviews, way more than any actual game. The first real game is SimCity (Limited Edition).

This matters because reviews of a gift card ("Works great, code arrived fast") are about delivery or convenience, not about gameplay. So a big chunk of this "video game" dataset isn't really about video games.

In [ ]:
br = r"<br\s*/?>"

before = df["review_body"].str.contains(br, case=False, na=False).sum()

# remove empty reviews
df = df.dropna(subset=["review_body"])
df = df[df["review_body"].str.strip() != ""].copy()

# replace <br /> tags with a space
df["review_body"] = df["review_body"].str.replace(br, " ", case=False, regex=True)

after = df["review_body"].str.contains(br, case=False, na=False).sum()
print("Reviews with line breaks before:", before)
print("Reviews with line breaks after:", after)
print("Rows left:", len(df))
print()

for i, row in df.head(5).iterrows():
    print(f"--- {row['star_rating']} stars: {row['product_title']}")
    print(row["review_body"])
    print()

Before cleaning, 25,699 reviews had `<br />` line breaks. After cleaning, 0 do. Only 3 empty reviews were removed (145,431 down to 145,428 rows).

What still looks weird:
- Missing spaces after periods ("saying much.The game"), where the line break was right after a period.
- Extra double spaces where the tags used to be.
- Slang and abbreviations ("PITA"), typos ("This years version"), and ALL CAPS ("NOT TO BUY THE MADDEN BUNDLE").
- Very short reviews ("Awesome", "Perfect") and inside jokes ("Hail to the great Yuri!") that need context to understand.

I don't think the spacing problems would change whether a model reads a review as positive or negative, since the words are still there. The bigger problems are things like slang, sarcasm, mixed opinions (the Madden review says some good things but is 2 stars), and jokes that need outside knowledge.

In [ ]:
sample = df.groupby("star_rating").sample(n=100, random_state=42).reset_index(drop=True)

# 20 extra one-star reviews for Part 5, not already in the sample
one_star = df[(df["star_rating"] == 1) & (~df["review_id"].isin(sample["review_id"]))]
complaints = one_star.sample(n=20, random_state=42).reset_index(drop=True)

print(sample["star_rating"].value_counts().sort_index())
print("Sample size:", len(sample))
print("Complaints size:", len(complaints))

We take the same number of each star rating because the full data is very unbalanced (55.5% are 5 stars, only 5.3% are 2 stars). A random sample of 500 would be about 277 five-star reviews and only about 26 two-star reviews. That would make the model look good just by getting the easy positive reviews right, and we wouldn't have enough 2- and 3-star reviews to tell how well it handles negative or mixed reviews. A balanced sample lets us test the model fairly on every rating.

In [ ]:
from transformers import pipeline

sentiment = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
)

review = sample.iloc[0]
print("Customer gave:", review["star_rating"], "stars")
print(review["review_body"])
print()

result = sentiment(review["review_body"], truncation=True)
result

The result is a list with one dictionary that has two keys: `label` and `score`. The label is the model's prediction (POSITIVE or NEGATIVE), and the score is how confident the model is in that label, from 0 to 1. Here it's NEGATIVE with a score of 0.9998, so the model is about 99.98% sure the review is negative.

The model agrees with the customer: the customer gave 1 star and the model said NEGATIVE.

In [ ]:
%%time
from tqdm.auto import tqdm

texts = sample["review_body"].tolist()
results = [sentiment(t, truncation=True)[0] for t in tqdm(texts)]

sample["label"] = [r["label"] for r in results]
sample["score"] = [r["score"] for r in results]

sample[["star_rating", "label", "score"]].head(10)

500 reviews took about 74 seconds, which is roughly 0.15 seconds per review (about 7 reviews per second). At that rate, all 145,428 reviews would take about 21,500 seconds, which is around 6 hours.

`truncation=True` protects against reviews that are too long for the model. This model can only read up to 512 tokens (roughly 300–400 words) at a time. Some reviews are over 5,000 words, and without truncation the model would crash with an error on them. With truncation, it just cuts the review off at 512 tokens and only reads the beginning, which means it might miss something important at the end of a long review.

In [ ]:
# answer key: 1-2 stars = NEGATIVE, 4-5 stars = POSITIVE, 3 stars left out
scored = sample[sample["star_rating"] != 3].copy()
scored["truth"] = scored["star_rating"].apply(lambda s: "POSITIVE" if s >= 4 else "NEGATIVE")
scored["correct"] = scored["label"] == scored["truth"]

print("Reviews scored:", len(scored))
print("Correct:", scored["correct"].sum())
print(f"Accuracy: {scored['correct'].mean():.1%}")

We leave out the 3-star reviews because they don't have a clear right answer. A 3-star review is neutral or mixed, but this model can only say POSITIVE or NEGATIVE, so there's no fair way to say whether its answer is right or wrong.

The accuracy is 87.8% (351 out of 400). That's pretty good for a model that was never trained on video game reviews (it was trained on movie reviews). But I wouldn't fully trust it in a product without a human checking: about 1 in 8 reviews gets the wrong label, it can't handle neutral reviews at all, and this was measured on easy cases (we removed the hard 3-star reviews). It would be fine for spotting general trends across lots of reviews, but not for making decisions about individual reviews.

In [ ]:
breakdown = pd.crosstab(sample["star_rating"], sample["label"])
breakdown["% POSITIVE"] = (breakdown["POSITIVE"] / breakdown.sum(axis=1) * 100).round(0)
breakdown

The model gets 4-star reviews wrong most often: it called 23 of the 100 four-star reviews NEGATIVE. 5-star reviews were next with 17 wrong. It does very well on 1- and 2-star reviews (only 4 and 5 wrong). So the model is better at spotting unhappy customers than happy ones. It probably gets confused when a positive review also mentions some complaints, like "great game, but the servers lag."

For the 3-star reviews, the model was forced to pick a side, since it only knows POSITIVE and NEGATIVE. It called 66 of them NEGATIVE and 34 POSITIVE, so it leans negative. That makes sense because mixed reviews usually spend a lot of words on what's wrong, and the model reacts to those negative words.

In [ ]:
mistakes = scored[~scored["correct"]].sort_values("score", ascending=False).head(5)

for i, row in mistakes.iterrows():
    print(f"--- Customer: {row['star_rating']} stars | Model: {row['label']} ({row['score']:.3f})")
    print(row["review_body"][:600])
    print()

1. **1 star, "Enjoys game" → POSITIVE:** The model is right here. The text is clearly positive, so the customer probably clicked the wrong star rating by mistake. **The rating is misleading.**

2. **4 stars, "I am not enjoying this game..." → NEGATIVE:** The text really is mostly negative (stuck, hint button not helpful). The customer gave 4 stars anyway, maybe because they like this type of game overall. **The rating is misleading** — the model read the words correctly.

3. **2 stars, "very difficult to install... quite enjoyable... fun" → POSITIVE:** This is a mixed review. The customer's low rating is about the install problems, but the review ends on positive words ("enjoyable", "fun"), which the model picked up. **Both** — it's a genuinely mixed review, and the model can't weigh which part mattered more to the customer.

4. **5 stars, "PROTIP: DO NOT DOWNLOAD FROM AMAZON!..." → NEGATIVE:** The customer liked the game but was warning people about Amazon's outdated version. The model saw "DO NOT", "unfortunately", and "bugs" and called it negative. **This is the model's mistake** — it doesn't understand that the complaint is about where to buy it, not the game itself.

5. **4 stars, "Its ok not great" → NEGATIVE:** The text is lukewarm at best, which sounds more like 3 stars. The model reacted to "not great." **The rating is misleading** (a bit generous), and the model only has two choices, so lukewarm gets pushed to NEGATIVE.

Overall, most of these mistakes come from the customer's star rating not matching their text, not from the model being bad. This shows star ratings aren't a perfect answer key.

In [ ]:
%%time
stars_model = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
)

star_results = [stars_model(t, truncation=True)[0] for t in tqdm(texts)]

sample["pred_stars"] = [int(r["label"][0]) for r in star_results]
sample["diff"] = (sample["pred_stars"] - sample["star_rating"]).abs()

print(f"Exact match: {(sample['diff'] == 0).mean():.1%}")
print(f"Within one star: {(sample['diff'] <= 1).mean():.1%}")
pd.crosstab(sample["star_rating"], sample["pred_stars"], rownames=["actual"], colnames=["predicted"])

The star model gets the exact star rating 57.4% of the time, and is within one star 91.2% of the time. Looking at the table, most of its mistakes are "neighbor" mistakes, like calling a 2-star review 1 star or a 5-star review 4 stars, which are small and understandable. It almost never mixes up a really bad review with a really good one.

For a sentiment dashboard for a game publisher, I would use the star model (nlptown). It was trained on product reviews, which are closer to game reviews than the movie reviews the first model learned from. More importantly, it can give a middle answer (3 stars) for mixed or "it's ok" reviews instead of being forced to call them positive or negative, and it shows *how* happy or unhappy customers are, not just which side they're on. That's more useful for spotting trends on a dashboard. The downsides are that it's about twice as slow (2 min 30 s vs 1 min 14 s for 500 reviews), and if the publisher only needed a simple positive/negative split, the first model was more accurate at that (87.8%).

In [ ]:
%%time
zero_shot = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

labels = [
    "bugs and crashes",
    "download or installation problems",
    "price or value",
    "boring gameplay",
    "account or code problems",
]

zs_results = [zero_shot(t, candidate_labels=labels) for t in tqdm(complaints["review_body"].tolist())]

complaints["topic"] = [r["labels"][0] for r in zs_results]
complaints["topic_score"] = [r["scores"][0] for r in zs_results]

complaints["topic"].value_counts()

Customers complain most about download or installation problems (6 of 20), followed closely by account or code problems (5 of 20). Together that's over half the complaints, and both are about *getting* the game to work, not the game itself. That fits with what we saw earlier: a lot of this dataset is digital codes, gift cards, and subscriptions, so delivery problems are a big deal. Price/value (4), boring gameplay (3), and bugs/crashes (2) were less common.

In [ ]:
for i, row in complaints.head(3).iterrows():
    print(f"--- Topic: {row['topic']} ({row['topic_score']:.2f})")
    print(row["review_body"][:500])
    print()

Mostly, yes:
- **"Never got the code, was still charged" → account or code problems (0.64):** Makes total sense, and it's the model's most confident pick.
- **"We never got this program to run properly!..." → bugs and crashes (0.39):** Reasonable. It's about the game not working, though "download or installation problems" would have fit too.
- **"Didn't order it" → account or code problems (0.35):** Kind of makes sense (sounds like an account/billing issue), but none of the labels really fit, so the model just picked the closest one.

The scores are low (0.35–0.64), which means the model often isn't very sure. That's because the reviews are short and some don't match any of my categories well. The model always has to pick one, even when none fit.

In [ ]:
%%time
new_labels = labels + [
    "technical problems",   # overlaps with "bugs and crashes"
    "cooking recipes",      # irrelevant
]

first5 = complaints.head(5)["review_body"].tolist()
new_results = [zero_shot(t, candidate_labels=new_labels) for t in tqdm(first5)]

compare = pd.DataFrame({
    "review": [t[:60] for t in first5],
    "old_top": complaints.head(5)["topic"].tolist(),
    "old_score": complaints.head(5)["topic_score"].round(2).tolist(),
    "new_top": [r["labels"][0] for r in new_results],
    "new_score": [round(r["scores"][0], 2) for r in new_results],
    "cooking_score": [round(dict(zip(r["labels"], r["scores"]))["cooking recipes"], 3) for r in new_results],
})
pd.set_option("display.max_colwidth", 80)
compare

**What happened:** Review 1 ("We never got this program to run properly") switched from "bugs and crashes" to the new overlapping label "technical problems." The other four kept the same top label, but their scores mostly dropped a bit (for example 0.35 → 0.29 and 0.64 → 0.58). The irrelevant "cooking recipes" label never won, but it still took a small share of the score every time (up to 0.027).

**Why (scores add up to 1):** Since every review's scores must add up to 1, each new label has to take some probability away from the old ones. The overlapping label "technical problems" means almost the same thing as "bugs and crashes," so they split the votes for that kind of review, and it even won one. Even "cooking recipes," which makes no sense, gets a little probability, so every other label's score shrinks slightly. The scores are relative ("which of these fits best?"), not absolute ("how well does this fit?").

**What it means for a real product:** The labels you choose change the results. Labels should be clearly different from each other with no overlap, or the counts get split in confusing ways. You should only include labels that are actually relevant, and you should test the labels on real reviews and read the results before trusting them. Also, a low top score is a sign that none of the labels fit well, so it might be worth adding an "other" category.